In [0]:
%pip install --quiet --upgrade databricks-sdk>=0.74.0 psycopg
%restart_python

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import (
    SyncedDatabaseTable,
    SyncedTableSpec,
    SyncedTableSchedulingPolicy,
    NewPipelineSpec,
)

w = WorkspaceClient()

In [0]:
INSTANCE_NAME = "icu-step-down"
TABLES = {
    'admissions': ['ROW_ID'], 
    'callout': ['ROW_ID'], 
    'caregivers': ['ROW_ID'], 
    'chart_events': ['ROW_ID'], 
    'cpt_events': ['ROW_ID'], 
    'd_icd_diagnoses': ['ROW_ID'], 
    'd_icd_procedures': ['ROW_ID'], 
    'd_labitems': ['ROW_ID'], 
    'diagnoses_icd': ['ROW_ID'], 
    'drg_codes': ['ROW_ID'], 
    'icu_stays': ['ROW_ID'], 
    'input_events_cv': ['ROW_ID'], 
    'input_events_mv': ['ROW_ID'], 
    'lab_events': ['ROW_ID'], 
    'microbiology_events': ['ROW_ID'], 
    'note_events': ['ROW_ID'], 
    'output_events': ['ROW_ID'], 
    'patients': ['ROW_ID'], 
    'prescriptions': ['ROW_ID'], 
    'procedure_events_mv': ['ROW_ID'], 
    'procedures_icd': ['ROW_ID'], 
    'services': ['ROW_ID'], 
    'transfers': ['ROW_ID']
}

PG_DATABASE = "mimic_iii"
SYNCED_UC_CATALOG = "users"
SYNCED_UC_SCHEMA = "mimic_iii"
existing_pipeline_id = "b9ad9c9a-a93f-4ec0-af57-ab32e0452ec3"

In [0]:
for table_name, _ in TABLES.items():
    print(f"Loading {table_name}")
    spark.sql(f"INSERT INTO users.mimic_iii.{table_name}_ SELECT * FROM mimic_iii_field_eng_east.mimic_iii.{table_name}")

In [0]:
for table_name, _ in TABLES.items():
    print(f"Loading {table_name}")
    spark.sql(f"SELECT * FROM users.mimic_iii.{table_name}_").show(5, False)

In [0]:
dst_tables = w.tables.list(catalog_name="users", schema_name="mimic_iii")
dst_table_names = {t.name for t in dst_tables}

TABLES = {name: columns for name, columns in TABLES.items() if name not in dst_table_names}

print(TABLES)

In [0]:
src_tables = w.tables.list(catalog_name="users", schema_name="mimic_iii")

src_table_names = {t.name: ["ROW_ID"] for t in src_tables}

print(src_table_names)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6715594291192001>, line 1
----> 1 src_tables = w.tables.list(catalog_name="users", schema_name="mimic_iii")
      3 src_table_names = {t.name: ["ROW_ID"] for t in src_tables}
      5 print(src_table_names)

NameError: name 'w' is not defined

In [0]:
pipeline_id = None

for table, columns in TABLES.items():
    SRC_TABLE_FULL_NAME = f"users.mimic_iii.{table}_"
    SYNCED_UC_FULL_NAME = f"{SYNCED_UC_CATALOG}.{SYNCED_UC_SCHEMA}.{table}"
    PRIMARY_KEY_COLUMNS = columns

    print(SYNCED_UC_FULL_NAME)
    print(",".join(columns))

    spark.sql(f"SELECT * FROM {SRC_TABLE_FULL_NAME} LIMIT 5").collect()

    if pipeline_id:
        spec = SyncedTableSpec(
            source_table_full_name=SRC_TABLE_FULL_NAME,
            primary_key_columns=PRIMARY_KEY_COLUMNS,
            scheduling_policy=SyncedTableSchedulingPolicy.SNAPSHOT,
            create_database_objects_if_missing=True,  # Create database/schema if needed
            # Optional: timeseries_key="timestamp"
            existing_pipeline_id=pipeline_id
        )
    else:
        spec=SyncedTableSpec(
            source_table_full_name=SRC_TABLE_FULL_NAME,
            primary_key_columns=PRIMARY_KEY_COLUMNS,
            scheduling_policy=SyncedTableSchedulingPolicy.SNAPSHOT,
            create_database_objects_if_missing=True,  # Create database/schema if needed
            # Optional: timeseries_key="timestamp"
            new_pipeline_spec=NewPipelineSpec(
                storage_catalog=SYNCED_UC_CATALOG, #"users",
                storage_schema=SYNCED_UC_SCHEMA #"matt_slack"
            )
        )

    synced_table = w.database.create_synced_database_table(
        SyncedDatabaseTable(
            name=SYNCED_UC_FULL_NAME,
            database_instance_name=INSTANCE_NAME,
            logical_database_name=PG_DATABASE,
            spec=spec
        )
    )

    pipeline_id = synced_table.data_synchronization_status.pipeline_id

    # InvalidParameterValue: Table without columns are not supported in online materialized views.

In [0]:
from databricks.sdk.service.catalog import TableType

tables = w.tables.list(catalog_name="users", schema_name="mimic_iii")

table_names = [t.name for t in tables if t.table_type == TableType.FOREIGN]

print(table_names)

In [0]:
from databricks.sdk.errors import NotFound

for table in table_names:
    SYNCED_UC_FULL_NAME = f"{SYNCED_UC_CATALOG}.{SYNCED_UC_SCHEMA}.{table}"

    print(f"Deleting {SYNCED_UC_FULL_NAME}...")

    try:
        w.database.delete_synced_database_table(SYNCED_UC_FULL_NAME)

        print(f"Table {SYNCED_UC_FULL_NAME} has been deleted.")
    except NotFound as e:
        print(f"Table {SYNCED_UC_FULL_NAME} does not exist.")

In [0]:
import psycopg
import uuid

instance = w.database.get_database_instance(INSTANCE_NAME)
cred = w.database.generate_database_credential(request_id=str(uuid.uuid4()), instance_names=[INSTANCE_NAME])

current_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .getOrElse(None)
)

conn_conf = {
    "host": instance.read_write_dns,
    "dbname": PG_DATABASE,
    "user": current_user,
    "password": cred.token,
    "sslmode": "require",
    "autocommit": True,
}

In [0]:
with psycopg.connect(**conn_conf) as conn:
    with conn.cursor() as cur:
        cur.execute(f"select table_name from information_schema.tables where table_schema = '{SYNCED_UC_SCHEMA}'")
        tables = [row[0] for row in cur.fetchall()]

print(tables)

In [0]:
with psycopg.connect(**conn_conf) as conn:
    with conn.cursor() as cur:
        for table_name in tables:
            print(f"Deleting {SYNCED_UC_SCHEMA}.{table_name}...")

            try:
                cur.execute(f"DROP TABLE IF EXISTS {SYNCED_UC_SCHEMA}.{table_name}")

                print(f"Table {SYNCED_UC_SCHEMA}.{table_name} has been deleted.")
            except Exception as e:
                print(f"{e} Table {SYNCED_UC_SCHEMA}.{table_name} does not exist.")

In [0]:
with psycopg.connect(**conn_conf) as conn:
    with conn.cursor() as cur:
        cur.execute(f'grant usage on schema {SYNCED_UC_SCHEMA} to "17da3210-25b6-45d2-81e0-ebdf806d1832"')

        for table_name in tables:
            cur.execute(f'grant select on table {SYNCED_UC_SCHEMA}.{table_name} to "17da3210-25b6-45d2-81e0-ebdf806d1832"')